In [ ]:
import pandas as pd
import numpy as np
from IPython.display import display
import sys, os
import matplotlib.pyplot as plt
import seaborn as sns
%matplotlib inline
sys.path.append(os.path.abspath(os.path.join(os.getcwd(), '..')))

In [ ]:
df_train_raw = pd.read_csv('../data/raw/train.csv').drop('Id', axis=1)
df_train = df_train_raw.copy()
#df_train = df_train.drop('SalePrice', axis=1)  # labels deleted 
print(f'start dimensions --> {df_train_raw.shape}')
df_train_raw.head()

In [ ]:
#Analisis de datos faltantes
missing_data = df_train_raw.isnull().sum()
missing_data = pd.DataFrame(missing_data[missing_data > 0], columns=['missing_count'])
missing_data['missing_percentage (%)'] = np.round(missing_data['missing_count'] / df_train_raw.shape[0] * 100, 2)
missing_data = missing_data.sort_values(by='missing_count', ascending=False)
print('fields with missing data:')
missing_data

In [ ]:
#Manejo de datos faltantes
#Paso inicial eliminar las columnas que tienen mas del 85% de datos faltantes

#LotFrontage: La longitud lineal (en pies) de la calle que bordea la propiedad.
#Alley: El tipo de acceso al callejón.
#MasVnrType: El tipo de revestimiento de mampostería.
#FireplaceQu: La calidad de la chimenea.
#PoolQC: La calidad de la piscina.
#Fence: La calidad de la valla.
#MiscFeature: Una característica miscelánea no cubierta en otras categorías.

print(f'start dimentions {df_train.shape}')
initial_columns = df_train.columns
df_train = df_train.dropna(thresh=len(df_train) * 0.85, axis=1)
columns_deleted = [col for col in initial_columns if col not in df_train.columns]
print(f'columns deleted ({len(columns_deleted)}) --> {columns_deleted}')

In [ ]:
#Columnas con datos faltantes resultantes de la eliminacion anterior
#Definir en una funcion
def missing_data_report(df):
    missing_res_deleted = df.isnull().sum()
    missing_res_deleted = missing_res_deleted[missing_res_deleted > 0].sort_values(ascending=True)
    return missing_res_deleted
res = missing_data_report(df_train)
res

In [ ]:
#imputacion simple
#Electrical: El sistema eléctrico. -> Valores faltantes imputados con la moda
df_train['Electrical'] = df_train['Electrical'].fillna(df_train['Electrical'].mode()[0])
#MasVnrArea: Área de revestimiento de mampostería en pies cuadrados. -> Valores faltantes imputados con (0)
df_train['MasVnrArea'] = df_train['MasVnrArea'].fillna(0)

In [ ]:
cond_not_basement = (df_train['TotalBsmtSF'] == 0) & (df_train['BsmtUnfSF'] == 0)
#BsmtCond: Evalúa la condición general del sótano. -> (NA) no tiene sótano
#BsmtQual: Evalúa la altura del sótano. -> (NA) no tiene sótano
##BsmtFinType1: Evaluación del tipo de acabado del sótano. -> (NA) no tiene sótano
#BsmtExposure: Refleja la cantidad de exposición al sótano al aire exterior. -> (NA) no tiene sótano
#BsmtFinType2: Evaluación del tipo de acabado del sótano (si hay dos tipos). -> (NA) no tiene sótano
cols_not_basement = ['BsmtCond', 'BsmtQual', 'BsmtFinType1', 'BsmtExposure', 'BsmtFinType2']
df_train.loc[cond_not_basement, cols_not_basement] = df_train.loc[cond_not_basement, cols_not_basement].fillna('NA')
fig, ax = plt.subplots(ncols=2, nrows=3)
fig.set_size_inches(15,8)

axisOne = sns.violinplot(x='BsmtCond', y='TotalBsmtSF', data=df_train, cut=0, ax=ax[0][0])
axisOne.set_title('Distribución de TotalBsmtSF por BsmtCond')
axisOne.set_xlabel('BsmtCond')
axisOne.set_ylabel('TotalBsmtSF')
axisOne.grid(axis='y', linestyle='--');

axisTwo = sns.violinplot(x='BsmtQual', y='TotalBsmtSF', data=df_train, cut=0, ax=ax[0][1])
axisTwo.set_title('Distribución de TotalBsmtSF por BsmtQual')
axisTwo.set_xlabel('BsmtQual')
axisTwo.set_ylabel('TotalBsmtSF')
axisTwo.grid(axis='y', linestyle='--');

axisThree = sns.violinplot(x='BsmtFinType1', y='TotalBsmtSF', data=df_train, cut=0, ax=ax[1][0])
axisThree.set_title('Distribución de TotalBsmtSF por BsmtFinType1')
axisThree.set_xlabel('BsmtFinType1')
axisThree.set_ylabel('TotalBsmtSF')
axisThree.grid(axis='y', linestyle='--');

axisFour = sns.violinplot(x='BsmtExposure', y='TotalBsmtSF', data=df_train, cut=0, ax=ax[1][1])
axisFour.set_title('Distribución de TotalBsmtSF por BsmtExposure')
axisFour.set_xlabel('BsmtExposure')
axisFour.set_ylabel('TotalBsmtSF')
axisFour.grid(axis='y', linestyle='--');

axisFive = sns.violinplot(x='BsmtFinType2', y='TotalBsmtSF', data=df_train, cut=0, ax=ax[2][0])
axisFive.set_title('Distribución de TotalBsmtSF por BsmtFinType2')
axisFive.set_xlabel('BsmtFinType2')
axisFive.set_ylabel('TotalBsmtSF')
axisFive.grid(axis='y', linestyle='--');

fig.delaxes(ax[2][1])
plt.tight_layout()

In [ ]:
#Columnas con datos faltantes resultantes de la imputacion anterior

missing_data2 = df_train.isnull().sum().filter(like='Bsmt')
missing_data2 = pd.DataFrame(missing_data2[missing_data2 > 0], columns=['missing_count'])
display(missing_data2)

#BsmtExposure: Refleja la cantidad de exposición al sótano al aire exterior. (x)
#BsmtFinType2: Evaluación del tipo de acabado del sótano (si hay dos tipos). (x)
df_train['BsmtExposure'] = df_train['BsmtExposure'].fillna(df_train.groupby(['BsmtCond', 'BsmtFinType1', 'BsmtFinType2', 'BsmtQual', 'BsmtFullBath', 'BsmtHalfBath'])['BsmtExposure']
                                                    .transform(lambda x: x.mode()[0] if not x.mode().empty else 'No'))
df_train['BsmtFinType2'] = df_train['BsmtFinType2'].fillna(df_train.groupby(['BsmtCond', 'BsmtQual', 'BsmtFullBath', 'BsmtHalfBath', 'BsmtExposure'])['BsmtFinType2']
                                                    .transform(lambda x: x.mode()[0] if not x.mode().empty else 'Unf'))

In [ ]:
#Imputacion de datos faltantes en columnas relacionadas con el garage ['GarageType', 'GarageYrBlt', 'GarageFinish', 'GarageQual', 'GarageCond'] -> ['GarageArea', 'GarageCars']
#GarageType ->  Ubicación del garaje. (NA) no tiene garage
#GarageYrBlt -> Año de construcción del garaje. (0) no tiene garage
#GarageFinish -> Acabado interior del garaje. (NA) no tiene garage
#GarageQual -> Calidad del garaje. (NA) no tiene garage
#GarageCond -> Condición del garaje. (NA) no tiene garage
list_categorical_not_garage = ['GarageType', 'GarageFinish', 'GarageQual', 'GarageCond']
list_numerical_not_garage = ['GarageYrBlt']
cond_not_garage = df_train['GarageArea'] == 0
df_train.loc[cond_not_garage, list_categorical_not_garage] = df_train.loc[cond_not_garage, list_categorical_not_garage].fillna('NA')
df_train.loc[cond_not_garage, list_numerical_not_garage] = df_train.loc[cond_not_garage, list_numerical_not_garage].fillna(0)

In [ ]:
fig, ax = plt.subplots(ncols=2, nrows=2)
fig.set_size_inches(15,8)
axisType = sns.violinplot(x='GarageType', y='GarageArea', data=df_train, cut=0, ax=ax[0][0])
axisType.set_title('Distribución del área del garaje por tipo de garaje')
axisType.set_xlabel('GarageType')
axisType.set_ylabel('GarageArea')
axisType.grid(axis='y', linestyle='--');
axisFinish = sns.violinplot(x='GarageFinish', y='GarageArea', data=df_train, cut=0, ax=ax[0][1])
axisFinish.set_title('Distribución del área del garaje por acabado del garaje')
axisFinish.set_xlabel('GarageFinish')
axisFinish.set_ylabel('GarageArea')
axisFinish.grid(axis='y', linestyle='--');
axisQual = sns.violinplot(x='GarageQual', y='GarageArea', data=df_train, cut=0, ax=ax[1][0])
axisQual.set_title('Distribución del área del garaje por calidad del garaje')
axisQual.set_xlabel('GarageQual')
axisQual.set_ylabel('GarageArea')
axisQual.grid(axis='y', linestyle='--');
axisCond = sns.violinplot(x='GarageCond', y='GarageArea', data=df_train, cut=0, ax=ax[1][1])
axisCond.set_title('Distribución del área del garaje por condición del garaje')
axisCond.set_xlabel('GarageCond')
axisCond.set_ylabel('GarageArea')
axisCond.grid(axis='y', linestyle='--');
plt.tight_layout()


In [ ]:
#Feature engineering
print(f'Numero de features --> {len(df_train.columns)}')
numeric_features = df_train.select_dtypes(include=[np.number]).columns.tolist()
categorical_features = df_train.select_dtypes(include=[object]).columns.tolist()
print(f'numericas {len(numeric_features)} --> {numeric_features}')
print(f'categoricas {len(categorical_features)} --> {categorical_features}')

In [ ]:
#MSZoning: Clasificación general de la zona de uso del suelo.
print('---'*20)
mszoning_cat_all = ['A', 'C', 'FV', 'I', 'RH', 'RL', 'RP', 'RM']
mszoning_cat_train = df_train['MSZoning'].unique().tolist()
missing_cats = list(set(mszoning_cat_all) - set(mszoning_cat_train))
print(f'{"🟢" if len(missing_cats) == 0 else "🔴"} missing categories {missing_cats}')

mszzoning_new_cat = df_train[~df_train['MSZoning'].isin(mszoning_cat_all)]
mszzoning_new_cat = mszzoning_new_cat['MSZoning'].unique().tolist()
print(f'{"🟢" if len(mszzoning_new_cat) == 0 else "🔴"} new categories or diferent name -> {mszzoning_new_cat}')


#Considerar valores categoricos de MSZoning que no tienen ejemplos en el set de entrenamiento
#No se tiene un orden definido en las categorias
#Pocas categorias
#Visto que las categorias no tienen un orden definido, se procede a hacer one-hot-encoding

#1. Se reemplaza 'C (all)' por 'C'
df_train['MSZoning'] = df_train['MSZoning'].replace('C (all)', 'C')

mszoning_cat_after_engineering = df_train['MSZoning'].unique().tolist()
missing_cats = list(set(mszoning_cat_all) - set(mszoning_cat_after_engineering))
print(f'{"🟢" if len(missing_cats) == 0 else "🔴"} missing categories after engineering {missing_cats}')

print('🟡 MSZoning: --> One-Hot-Encoding')
print('---'*20)
ohe_mszoning = pd.get_dummies(df_train['MSZoning'], prefix='MSZoning', dtype=int)
display(ohe_mszoning.head())

In [ ]:
#Street: Tipo de acceso a la calle.
print('---'*20)

street_cat_all = ['Grvl', 'Pave']
street_cat_train = df_train['Street'].unique().tolist()
missing_cats = list(set(street_cat_all) - set(street_cat_train))
print(f'{"🟢" if len(missing_cats) == 0 else "🔴"} missing categories {missing_cats}')

street_new_cat = df_train[~df_train['Street'].isin(street_cat_all)]
street_new_cat = street_new_cat['Street'].unique().tolist()
print(f'{"🟢" if len(street_new_cat) == 0 else "🔴"} new categories or diferent name -> {street_new_cat}')
#No se tiene un orden definido en las categorias
#Pocas categorias
#Visto que las categorias no tienen un orden definido, se procede a hacer one-hot-encoding
#@@ Podria ser ordinal encoding
print('🟡 Street: --> One-Hot-Encoding')
print('---'*20)
ohe_street = pd.get_dummies(df_train['Street'], prefix='Street', dtype=int)
display(ohe_street.head())

In [ ]:
#LotShape: Forma de la parcela.
print('---'*20)
lotshape_cat_all = ['Reg', 'IR1', 'IR2', 'IR3']
lotshape_cat_train = df_train['LotShape'].unique().tolist()
missing_cats = list(set(lotshape_cat_all) - set(lotshape_cat_train))
print(f'{"🟢" if len(missing_cats) == 0 else "🔴"} missing categories {missing_cats}')

lotshape_new_cat = df_train[~df_train['LotShape'].isin(lotshape_cat_all)]
lotshape_new_cat = lotshape_new_cat['LotShape'].unique().tolist()
print(f'{"🟢" if len(lotshape_new_cat) == 0 else "🔴"} new categories or diferent name -> {lotshape_new_cat}')

#Se tiene un orden definido en las categorias
#Pocas categorias
#Visto que las categorias tienen un orden definido, se procede a hacer ordinal-encoding
#Preliminarmene se hara un mapeo estandar
print('🟡 LotShape: --> Ordinal Encoding')
print('---'*20)
mapping_lotshape = {'Reg': 0, 'IR1': 1, 'IR2': 2, 'IR3': 3}
display(mapping_lotshape)

In [ ]:
#LandContour: Plano de la parcela.
print('---'*20)
landcontour_cat_all = ['Lvl', 'Bnk', 'HLS', 'Low']
landcontour_cat_train = df_train['LandContour'].unique().tolist()
missing_cats = list(set(landcontour_cat_all) - set(landcontour_cat_train))
print(f'{"🟢" if len(missing_cats) == 0 else "🔴"} missing categories {missing_cats}')

landcontour_new_cat = df_train[~df_train['LandContour'].isin(landcontour_cat_all)]
landcontour_new_cat = landcontour_new_cat['LandContour'].unique().tolist()
print(f'{"🟢" if len(landcontour_new_cat) == 0 else "🔴"} new categories or diferent name -> {landcontour_new_cat}')
#Se tiene un orden definido en las categorias
#Pocas categorias
#Visto que las categorias tienen un orden definido, se procede a hacer ordinal-encoding
print('🟡 LandContour: --> Ordinal Encoding')
print('---'*20)
mapping_landcontour = {'Lvl': 0, 'Bnk': 1, 'HLS': 2, 'Low': 3}
display(mapping_landcontour)

In [ ]:
#Utilities: Tipo de servicios públicos disponibles.
print('---'*20)
utilities_cat_all = ['AllPub', 'NoSewr', 'NoSeWa', 'ELO']
utilities_cat_train = df_train['Utilities'].unique().tolist()
missing_cats = list(set(utilities_cat_all) - set(utilities_cat_train))
print(f'{"🟢" if len(missing_cats) == 0 else "🔴"} missing categories {missing_cats}')

utilities_new_cat = df_train[~df_train['Utilities'].isin(utilities_cat_all)]
utilities_new_cat = utilities_new_cat['Utilities'].unique().tolist()
print(f'{"🟢" if len(utilities_new_cat) == 0 else "🔴"} new categories or diferent name -> {utilities_new_cat}')

#Se tiene un orden definido en las categorias
#Pocas categorias
#Visto que las categorias tienen un orden definido, se procede a hacer ordinal-encoding
#@@@: Puede ser probada con one-hot-encoding
print('🟡 Utilities: --> Ordinal Encoding')
print('---'*20)
mapping_utilities = {'AllPub': 3, 'NoSewr': 2, 'NoSeWa': 1, 'ELO': 0}
display(mapping_utilities)

In [ ]:
#LotConfig: Configuración de la parcela.
print('---'*20)
print('LotConfig: --> one hot encoding')
lot_config_cat_all = ['Inside', 'Corner', 'CulDSac', 'FR2', 'FR3']
lot_config_cat_train = df_train['LotConfig'].unique().tolist()
missing_cats = list(set(lot_config_cat_all) - set(lot_config_cat_train))
print(f'{"🟢" if len(missing_cats) == 0 else "🔴"} missing categories {missing_cats}')

lotconfig_new_cat = df_train[~df_train['LotConfig'].isin(lot_config_cat_all)]
lotconfig_new_cat = lotconfig_new_cat['LotConfig'].unique().tolist()
print(f'{"🟢" if len(lotconfig_new_cat) == 0 else "🔴"} new categories or diferent name -> {lotconfig_new_cat}')

#No tiene un orden definido en las categorias
#Pocas categorias
#Visto que las categorias no tienen un orden definido, se procede a hacer one-hot-encoding
#@@@: Puede ser probada con ordinal-encoding
print('🟡 LotConfig: --> One-Hot-Encoding')
print('---'*20)

ohe_lotconfig = pd.get_dummies(df_train['LotConfig'], prefix='LotConfig', dtype=int)
display(ohe_lotconfig.head())

In [ ]:
#LandSlope: Inclinación del terreno.
print('---'*20)
landslope_cat_all = ['Gtl', 'Mod', 'Sev']
landslope_cat_train = df_train['LandSlope'].unique().tolist()
missing_cats = list(set(landslope_cat_all) - set(landslope_cat_train))
print(f'{"🟢" if len(missing_cats) == 0 else "🔴"} missing categories {missing_cats}')

landslope_new_cat = df_train[~df_train['LandSlope'].isin(landslope_cat_all)]
landslope_new_cat = landslope_new_cat['LandSlope'].unique().tolist()
print(f'{"🟢" if len(landslope_new_cat) == 0 else "🔴"} new categories or diferent name -> {landslope_new_cat}')
#Se tiene un orden definido en las categorias
#Pocas categorias
#Visto que las categorias tienen un orden definido, se procede a hacer ordinal-encoding
mapping_landslope = {'Gtl': 0, 'Mod': 1, 'Sev': 2}

print('🟡 LandSlope: --> One-Hot-Encoding')
print('---'*20)
ohe_landslope = pd.get_dummies(df_train['LandSlope'], prefix='LandSlope', dtype=int)
display(ohe_landslope.head())


In [ ]:
# Neighborhood: Ubicación física dentro de la ciudad de Ames.
print("---" * 20)
cat_all = ["Blmngtn","Blueste","BrDale	","BrkSide","ClearCr","CollgCr","Crawfor","Edwards","Gilbert","IDOTRR	","MeadowV","Mitchel","Names	","NoRidge","NPkVill","NridgHt","NWAmes	","OldTown","SWISU	","Sawyer	","SawyerW","Somerst","StoneBr","Timber	","Veenker"]
neighborhood_cat_all = [cat_all.strip() for cat_all in cat_all]
neighborhood_cat_train = df_train["Neighborhood"].unique().tolist()
missing_cats = list(set(neighborhood_cat_all) - set(neighborhood_cat_train))
print(f'{"🟢" if len(missing_cats) == 0 else "🔴"} missing categories {missing_cats}')

neighborhood_new_cat = df_train[~df_train["Neighborhood"].isin(neighborhood_cat_all)]
neighborhood_new_cat = neighborhood_new_cat["Neighborhood"].unique().tolist()
print(
    f'{"🟢" if len(neighborhood_new_cat) == 0 else "🔴"} new categories or diferent name -> {neighborhood_new_cat}'
)
# No se tiene un orden definido en las categorias
# Pocas categorias
# Visto que las categorias no tienen un orden definido, se procede a hacer one-hot-encoding
#1. Se reemplaza NAmes por Names
df_train["Neighborhood"] = df_train["Neighborhood"].replace("NAmes", "Names")

neighborhood_cat_after_engineering = df_train["Neighborhood"].unique().tolist()
missing_cats = list(set(neighborhood_cat_all) - set(neighborhood_cat_after_engineering))
print(f'{"🟢" if len(missing_cats) == 0 else "🔴"} missing categories after engineering {missing_cats}')

print("🟡 Neighborhood: --> One-Hot-Encoding")
print("---" * 20)
ohe_neighborhood = pd.get_dummies(
    df_train["Neighborhood"], prefix="Neighborhood", dtype=int
)
display(ohe_neighborhood.head())

In [ ]:
#Condition1: Proximidad a varias condiciones principales.
print('---'*20)
condition1_cat_all = [ "Artery", "Feedr", "Norm", "RRNn", "RRAn", "PosN", "PosA", "RRNe", "RRAe"]
condition1_cat_train = df_train['Condition1'].unique().tolist()
missing_cats = list(set(condition1_cat_all) - set(condition1_cat_train))
print(f'{"🟢" if len(missing_cats) == 0 else "🔴"} missing categories {missing_cats}')

condition1_new_cat = df_train[~df_train['Condition1'].isin(condition1_cat_all)]
condition1_new_cat = condition1_new_cat['Condition1'].unique().tolist()
print(f'{"🟢" if len(condition1_new_cat) == 0 else "🔴"} new categories or diferent name -> {condition1_new_cat}')
#No se tiene un orden definido en las categorias
#Pocas categorias
#Visto que las categorias no tienen un orden definido, se procede a hacer one-hot-encoding
#Faltan categorias en el set de entrenamiento
print('🟡 Condition1: --> One-Hot-Encoding')
print('---'*20)
ohe_condition1 = pd.get_dummies(df_train['Condition1'], prefix='Condition1', dtype=int)
display(ohe_condition1.head())

In [ ]:
#Condition2: Proximidad a varias condiciones secundarias.
print('---'*20)
condition2_cat_all = [ "Artery", "Feedr", "Norm", "RRNn", "RRAn", "PosN", "PosA", "RRNe", "RRAe"]
condition2_cat_train = df_train['Condition2'].unique().tolist()
missing_cats = list(set(condition2_cat_all) - set(condition2_cat_train))
print(f'{"🟢" if len(missing_cats) == 0 else "🔴"} missing categories {missing_cats}')


condition2_new_cat = df_train[~df_train['Condition2'].isin(condition2_cat_all)]
condition2_new_cat = condition2_new_cat['Condition2'].unique().tolist()
print(f'{"🟢" if len(condition2_new_cat) == 0 else "🔴"} new categories or diferent name -> {condition2_new_cat}')
#No se tiene un orden definido en las categorias
#Pocas categorias
#Visto que las categorias no tienen un orden definido, se procede a hacer one-hot-encoding
#Faltan categorias en el set de entrenamiento
print('🟡 Condition2: --> One-Hot-Encoding')
print('---'*20)
ohe_condition2 = pd.get_dummies(df_train['Condition2'], prefix='Condition2', dtype=int)
display(ohe_condition2.head())

In [ ]:
#BldgType : Tipo de vivienda.
print('---'*20)
bldgtype_cat_all = [ "1Fam", "2FmCon", "Duplx", "TwnhsE", "TwnhsI"]
bldgtype_cat_train = df_train['BldgType'].unique().tolist()
missing_cats = list(set(bldgtype_cat_all) - set(bldgtype_cat_train))
print(f'{"🟢" if len(missing_cats) == 0 else "🔴"} missing categories {missing_cats}')

bldgtype_new_cat = df_train[~df_train['BldgType'].isin(bldgtype_cat_all)]
bldgtype_new_cat = bldgtype_new_cat['BldgType'].unique().tolist()
print(f'{"🟢" if len(bldgtype_new_cat) == 0 else "🔴"} new categories or diferent name -> {bldgtype_new_cat}')

#No se tiene un orden definido en las categorias
#Pocas categorias
#Visto que las categorias no tienen un orden definido, se procede a hacer one-hot-encoding
#1. Convertir categorica Twnhs a TwnhsI
df_train['BldgType'] = df_train['BldgType'].replace('Twnhs', 'TwnhsI')
#2. Convertir categorica Duplx a Duplex
df_train['BldgType'] = df_train['BldgType'].replace('Duplex', 'Duplx')
#3. Convertir categorica 2fmCon a 2FmCon
df_train['BldgType'] = df_train['BldgType'].replace('2fmCon', '2FmCon')

bldgtype_cat_train = df_train['BldgType'].unique().tolist()
missing_cats = list(set(bldgtype_cat_all) - set(bldgtype_cat_train))
print(f'{"🟢" if len(missing_cats) == 0 else "🔴"} missing categories after engineering{missing_cats}')

print('🟡 BldgType: --> One-Hot-Encoding')
print('---'*20)
ohe_bldgtype = pd.get_dummies(df_train['BldgType'], prefix='BldgType', dtype=int)
display(ohe_bldgtype.head())

In [ ]:
#HouseStyle : Estilo de la casa.
print('---'*20)
housestyle_cat_all = [ "1Story", "1.5Fin", "1.5Unf", "2Story", "2.5Fin", "2.5Unf", "SFoyer", "SLvl"]
housestyle_cat_train = df_train['HouseStyle'].unique().tolist()
missing_cats = list(set(housestyle_cat_all) - set(housestyle_cat_train))
print(f'{"🟢" if len(missing_cats) == 0 else "🔴"} missing categories {missing_cats}')

housestyle_new_cat = df_train[~df_train['HouseStyle'].isin(housestyle_cat_all)]
housestyle_new_cat = housestyle_new_cat['HouseStyle'].unique().tolist()
print(f'{"🟢" if len(housestyle_new_cat) == 0 else "🔴"} new categories or diferent name -> {housestyle_new_cat}')
#No se tiene un orden definido en las categorias
#Pocas categorias
#Visto que las categorias no tienen un orden definido, se procede a hacer one-hot-encoding
#@@Analizar la posibilidad de aplicar ordinal encoding
print('🟡 HouseStyle: --> One-Hot-Encoding')
print('---'*20)
ohe_housestyle = pd.get_dummies(df_train['HouseStyle'], prefix='HouseStyle', dtype=int)
display(ohe_housestyle.head())

In [ ]:
#RoofStyle : Tipo de techo.
print('---'*20)
roofstyle_cat_all = ['Flat', 'Gable', 'Gambrel', 'Hip', 'Mansard', 'Shed']
roofstyle_cat_train = df_train['RoofStyle'].unique().tolist()
missing_cats = list(set(roofstyle_cat_all) - set(roofstyle_cat_train))
print(f'{"🟢" if len(missing_cats) == 0 else "🔴"} missing categories {missing_cats}')

roofstyle_new_cat = df_train[~df_train['RoofStyle'].isin(roofstyle_cat_all)]
roofstyle_new_cat = roofstyle_new_cat['RoofStyle'].unique().tolist()
print(f'{"🟢" if len(roofstyle_new_cat) == 0 else "🔴"} new categories or diferent name -> {roofstyle_new_cat}')
print('🟡 RoofStyle: --> One-Hot-Encoding')
print('---'*20)
#No se tiene un orden definido en las categorias
#Pocas categorias
#Visto que las categorias no tienen un orden definido, se procede a hacer one-hot-encoding
ohe_roofstyle = pd.get_dummies(df_train['RoofStyle'], prefix='RoofStyle', dtype=int)
display(ohe_roofstyle.head())

In [ ]:
#RoofMatl : Material del techo.
print('---'*20)
roofmatl_cat_all = [ "ClyTile", "CompShg", "Membran", "Metal", "Roll", "Tar&Grv", "WdShake", "WdShngl"]
roofmatl_cat_train = df_train['RoofMatl'].unique().tolist()
missing_cats = list(set(roofmatl_cat_all) - set(roofmatl_cat_train))
print(f'{"🟢" if len(missing_cats) == 0 else "🔴"} missing categories {missing_cats}')

roofmatl_new_cat = df_train[~df_train['RoofMatl'].isin(roofmatl_cat_all)]
roofmatl_new_cat = roofmatl_new_cat['RoofMatl'].unique().tolist()
print(f'{"🟢" if len(roofmatl_new_cat) == 0 else "🔴"} new categories or diferent name -> {roofmatl_new_cat}')
#No se tiene un orden definido en las categorias
#Pocas categorias
#Visto que las categorias no tienen un orden definido, se procede a hacer one-hot-encoding
print('🟡 RoofMatl: --> One-Hot-Encoding')
print('---'*20)

ohe_roofmatl = pd.get_dummies(df_train['RoofMatl'], prefix='RoofMatl', dtype=int)
display(ohe_roofmatl.head())

In [ ]:
#Exterior1st : Material exterior en la pared principal.
print('---'*20)
exterior1st_cat_all = [ "AsbShng", "AsphShn", "BrkComm", "BrkFace", "CBlock", "CemntBd", "HdBoard", "ImStucc", "MetalSd", "Other", "Plywood", "PreCast", "Stone", "Stucco", "VinylSd", "Wd Sdng", "WdShing"]
exterior1st_cat_train = df_train['Exterior1st'].unique().tolist()
missing_cats = list(set(exterior1st_cat_all) - set(exterior1st_cat_train))
print(f'{"🟢" if len(missing_cats) == 0 else "🔴"} missing categories {missing_cats}')

exterior1st_new_cat = df_train[~df_train['Exterior1st'].isin(exterior1st_cat_all)]
exterior1st_new_cat = exterior1st_new_cat['Exterior1st'].unique().tolist()
print(f'{"🟢" if len(exterior1st_new_cat) == 0 else "🔴"} new categories or diferent name -> {exterior1st_new_cat}')
#No se tiene un orden definido en las categorias
#Pocas categorias
#Visto que las categorias no tienen un orden definido, se procede a hacer one-hot-encoding
print('🟡 Exterior1st: --> One-Hot-Encoding')
print('---'*20)
ohe_exterior1st = pd.get_dummies(df_train['Exterior1st'], prefix='Exterior1st', dtype=int)
display(ohe_exterior1st.head())

In [ ]:
#Exterior2nd : Material exterior en la pared secundaria.
print('---'*20)
exterior2nd_cat_all = [ "AsbShng", "AsphShn", "BrkComm", "BrkFace", "CBlock", "CemntBd", "HdBoard", "ImStucc", "MetalSd", "Other", "Plywood", "PreCast", "Stone", "Stucco", "VinylSd", "Wd Sdng", "WdShing"]
exterior2nd_cat_train = df_train['Exterior2nd'].unique().tolist()
missing_cats = list(set(exterior2nd_cat_all) - set(exterior2nd_cat_train))
print(f'{"🟢" if len(missing_cats) == 0 else "🔴"} missing categories {missing_cats}')

exterior2nd_new_cat = df_train[~df_train['Exterior2nd'].isin(exterior2nd_cat_all)]
exterior2nd_new_cat = exterior2nd_new_cat['Exterior2nd'].unique().tolist()
print(f'{"🟢" if len(exterior2nd_new_cat) == 0 else "🔴"} new categories or diferent name -> {exterior2nd_new_cat}')
#No se tiene un orden definido en las categorias
#Pocas categorias
#Visto que las categorias no tienen un orden definido, se procede a hacer one-hot-encoding
#1. Se reemplaza Wd Shng por WdShing
df_train['Exterior2nd'] = df_train['Exterior2nd'].replace('Wd Shng', 'WdShing')
#2. Se reemplaza CmentBd por CmentBd por CemntBd
df_train['Exterior2nd'] = df_train['Exterior2nd'].replace('CmentBd', 'CemntBd')
#3. Se reemplaza Brk Cmn por BrkComm
df_train['Exterior2nd'] = df_train['Exterior2nd'].replace('Brk Cmn', 'BrkComm')

exterior2nd_cat_train = df_train['Exterior2nd'].unique().tolist()
missing_cats = list(set(exterior2nd_cat_all) - set(exterior2nd_cat_train))
print(f'{"🟢" if len(missing_cats) == 0 else "🔴"} missing categories after engineering {missing_cats}')

print('🟡 Exterior2nd: --> One-Hot-Encoding')
print('---'*20)
ohe_exterior2nd = pd.get_dummies(df_train['Exterior2nd'], prefix='Exterior2nd', dtype=int)
display(ohe_exterior2nd.head())

In [ ]:
#ExterQual : Evaluación de la calidad del material exterior.
print('---'*20)
exterqual_cat_all = ['Ex', 'Gd', 'TA', 'Fa', 'Po']
exterqual_cat_train = df_train['ExterQual'].unique().tolist()
missing_cats = list(set(exterqual_cat_all) - set(exterqual_cat_train))
print(f'{"🟢" if len(missing_cats) == 0 else "🔴"} missing categories {missing_cats}')

exterqual_new_cat = df_train[~df_train['ExterQual'].isin(exterqual_cat_all)]
exterqual_new_cat = exterqual_new_cat['ExterQual'].unique().tolist()
print(f'{"🟢" if len(exterqual_new_cat) == 0 else "🔴"} new categories or diferent name -> {exterqual_new_cat}')

#Se define como ordinal encoding
#Pocas categorias
mapping_exterqual = {'Ex': 4, 'Gd': 3, 'TA': 2, 'Fa': 1, 'Po': 0}
print('🟡 ExterQual: --> Ordinal Encoding')
print('---'*20)
display(mapping_exterqual)

In [ ]:
#ExterCond : Evaluación de la condición del material exterior.
print('---'*20)
extercond_cat_all = ['Ex', 'Gd', 'TA', 'Fa', 'Po']
extercond_cat_train = df_train['ExterCond'].unique().tolist()
missing_cats = list(set(extercond_cat_all) - set(extercond_cat_train))
print(f'{"🟢" if len(missing_cats) == 0 else "🔴"} missing categories {missing_cats}')

extercond_new_cat = df_train[~df_train['ExterCond'].isin(extercond_cat_all)]
extercond_new_cat = extercond_new_cat['ExterCond'].unique().tolist()
print(f'{"🟢" if len(extercond_new_cat) == 0 else "🔴"} new categories or diferent name -> {extercond_new_cat}')

#Se define como ordinal encoding
#Pocas categorias
mapping_extercond = {'Ex': 4, 'Gd': 3, 'TA': 2, 'Fa': 1, 'Po': 0}
print('🟡 ExterCond: --> Ordinal Encoding')
print('---'*20)
display(mapping_extercond)

In [ ]:
#Foundation : Tipo de cimiento.
print('---'*20)
foundation_cat_all = [ "BrkTil", "CBlock", "PConc", "Slab", "Stone", "Wood"]
foundation_cat_train = df_train['Foundation'].unique().tolist()
missing_cats = list(set(foundation_cat_all) - set(foundation_cat_train))
print(f'{"🟢" if len(missing_cats) == 0 else "🔴"} missing categories {missing_cats}')

foundation_new_cat = df_train[~df_train['Foundation'].isin(foundation_cat_all)]
foundation_new_cat = foundation_new_cat['Foundation'].unique().tolist()
print(f'{"🟢" if len(foundation_new_cat) == 0 else "🔴"} new categories or diferent name -> {foundation_new_cat}')
#No se tiene un orden definido en las categorias
#Pocas categorias
#Visto que las categorias no tienen un orden definido, se procede a hacer one-hot-encoding
print('🟡 Foundation: --> One-Hot-Encoding')
print('---'*20)
ohe_foundation = pd.get_dummies(df_train['Foundation'], prefix='Foundation', dtype=int)
display(ohe_foundation.head())

In [ ]:
#BsmtQual : Evalúa la altura del sótano.
print('---'*20)
bsmtqual_cat_all = ['Ex', 'Gd', 'TA', 'Fa', 'Po', 'NA']
bsmtqual_cat_train = df_train['BsmtQual'].unique().tolist()
missing_cats = list(set(bsmtqual_cat_all) - set(bsmtqual_cat_train))
print(f'{"🟢" if len(missing_cats) == 0 else "🔴"} missing categories {missing_cats}')

bsmtqual_new_cat = df_train[~df_train['BsmtQual'].isin(bsmtqual_cat_all)]
bsmtqual_new_cat = bsmtqual_new_cat['BsmtQual'].unique().tolist()
print(f'{"🟢" if len(bsmtqual_new_cat) == 0 else "🔴"} new categories or diferent name -> {bsmtqual_new_cat}')

#Se define como ordinal encoding
#Pocas categorias
mapping_bsmtqual = {'Ex': 4, 'Gd': 3, 'TA': 2, 'Fa': 1, 'Po': 0, 'NA': 0}
print('🟡 BsmtQual: --> Ordinal Encoding')
print('---'*20)
display(mapping_bsmtqual)

In [ ]:
#BsmtCond : Evalúa la condición general del sótano
print('---'*20)
bsmtcond_cat_all = ['Ex', 'Gd', 'TA', 'Fa', 'Po', 'NA']
bsmtcond_cat_train = df_train['BsmtCond'].unique().tolist()
missing_cats = list(set(bsmtcond_cat_all) - set(bsmtcond_cat_train))
print(f'{"🟢" if len(missing_cats) == 0 else "🔴"} missing categories {missing_cats}')

bsmtcond_new_cat = df_train[~df_train['BsmtCond'].isin(bsmtcond_cat_all)]
bsmtcond_new_cat = bsmtcond_new_cat['BsmtCond'].unique().tolist()
print(f'{"🟢" if len(bsmtcond_new_cat) == 0 else "🔴"} new categories or diferent name -> {bsmtcond_new_cat}')

#Se define como ordinal encoding
#Pocas categorias
mapping_bsmtcond = {'Ex': 4, 'Gd': 3, 'TA': 2, 'Fa': 1, 'Po': 0, 'NA': 0}
print('🟡 BsmtCond: --> Ordinal Encoding')
print('---'*20)
display(mapping_bsmtcond)

In [ ]:
#BsmtExposure : Refleja la cantidad de exposición al sótano al aire exterior.
print('---'*20)
bsmtexposure_cat_all = ['Gd', 'Av', 'Mn', 'No', 'NA']
bsmtexposure_cat_train = df_train['BsmtExposure'].unique().tolist()
missing_cats = list(set(bsmtexposure_cat_all) - set(bsmtexposure_cat_train))
print(f'{"🟢" if len(missing_cats) == 0 else "🔴"} missing categories {missing_cats}')

bsmtexposure_new_cat = df_train[~df_train['BsmtExposure'].isin(bsmtexposure_cat_all)]
bsmtexposure_new_cat = bsmtexposure_new_cat['BsmtExposure'].unique().tolist()
print(f'{"🟢" if len(bsmtexposure_new_cat) == 0 else "🔴"} new categories or diferent name -> {bsmtexposure_new_cat}')

#Se define como ordinal encoding
#Pocas categorias
mapping_bsmtexposure = {'Gd': 3, 'Av': 2, 'Mn': 1, 'No': 0, 'NA': 0}
print('🟡 BsmtExposure: --> Ordinal Encoding')
print('---'*20)
display(mapping_bsmtexposure)

In [ ]:
#BsmtFinType1 : Calidad del acabado del sótano.
print('---'*20)
bsmtfintype1_cat_all = ['GLQ', 'ALQ', 'BLQ', 'Rec', 'LwQ', 'Unf', 'NA']
bsmtfintype1_cat_train = df_train['BsmtFinType1'].unique().tolist()
missing_cats = list(set(bsmtfintype1_cat_all) - set(bsmtfintype1_cat_train))
print(f'{"🟢" if len(missing_cats) == 0 else "🔴"} missing categories {missing_cats}')

bsmtfintype1_new_cat = df_train[~df_train['BsmtFinType1'].isin(bsmtfintype1_cat_all)]
bsmtfintype1_new_cat = bsmtfintype1_new_cat['BsmtFinType1'].unique().tolist()
print(f'{"🟢" if len(bsmtfintype1_new_cat) == 0 else "🔴"} new categories or diferent name -> {bsmtfintype1_new_cat}')

#Se define como ordinal encoding
#Pocas categorias
mapping_bsmtfintype1 = {'GLQ': 6, 'ALQ': 5, 'BLQ': 4, 'Rec': 3, 'LwQ': 2, 'Unf': 1, 'NA': 0}
print('🟡 BsmtFinType1: --> Ordinal Encoding')
print('---'*20)
display(mapping_bsmtfintype1)

In [ ]:
#BsmtFinType2 : Calidad del acabado del sótano (si hay dos áreas terminadas).
print('---'*20)
bsmtfintype2_cat_all = ['GLQ', 'ALQ', 'BLQ', 'Rec', 'LwQ', 'Unf', 'NA']
bsmtfintype2_cat_train = df_train['BsmtFinType2'].unique().tolist()
missing_cats = list(set(bsmtfintype2_cat_all) - set(bsmtfintype2_cat_train))
print(f'{"🟢" if len(missing_cats) == 0 else "🔴"} missing categories {missing_cats}')

bsmtfintype2_new_cat = df_train[~df_train['BsmtFinType2'].isin(bsmtfintype2_cat_all)]
bsmtfintype2_new_cat = bsmtfintype2_new_cat['BsmtFinType2'].unique().tolist()
print(f'{"🟢" if len(bsmtfintype2_new_cat) == 0 else "🔴"} new categories or diferent name -> {bsmtfintype2_new_cat}')

#Se define como ordinal encoding
#Pocas categorias
mapping_bsmtfintype2 = {'GLQ': 6, 'ALQ': 5, 'BLQ': 4, 'Rec': 3, 'LwQ': 2, 'Unf': 1, 'NA': 0}
print('🟡 BsmtFinType2: --> Ordinal Encoding')
print('---'*20)
display(mapping_bsmtfintype2)

In [ ]:
#Heating : Tipo de calefacción.
print('---'*20)
heating_cat_all = ['Floor', 'GasA', 'GasW', 'Grav', 'OthW', 'Wall']
heating_cat_train = df_train['Heating'].unique().tolist()
missing_cats = list(set(heating_cat_all) - set(heating_cat_train))
print(f'{"🟢" if len(missing_cats) == 0 else "🔴"} missing categories {missing_cats}')

heating_new_cat = df_train[~df_train['Heating'].isin(heating_cat_all)]
heating_new_cat = heating_new_cat['Heating'].unique().tolist()
print(f'{"🟢" if len(heating_new_cat) == 0 else "🔴"} new categories or diferent name -> {heating_new_cat}')

#No se tiene un orden definido en las categorias
#Pocas categorias
#Visto que las categorias no tienen un orden definido, se procede a hacer one-hot-encoding
print('🟡 Heating: --> One-Hot-Encoding')
print('---'*20)
ohe_heating = pd.get_dummies(df_train['Heating'], prefix='Heating', dtype=int)
display(ohe_heating.head())

In [ ]:
#HeatingQC : Evaluación de la calidad y el estado del sistema de calefacción.
print('---'*20)
heatingqc_cat_all = ['Ex', 'Gd', 'TA', 'Fa', 'Po']
heatingqc_cat_train = df_train['HeatingQC'].unique().tolist()
missing_cats = list(set(heatingqc_cat_all) - set(heatingqc_cat_train))
print(f'{"🟢" if len(missing_cats) == 0 else "🔴"} missing categories {missing_cats}')

heatingqc_new_cat = df_train[~df_train['HeatingQC'].isin(heatingqc_cat_all)]
heatingqc_new_cat = heatingqc_new_cat['HeatingQC'].unique().tolist()
print(f'{"🟢" if len(heatingqc_new_cat) == 0 else "🔴"} new categories or diferent name -> {heatingqc_new_cat}')

#Se define como ordinal encoding
#Pocas categorias
mapping_heatingqc = {'Ex': 4, 'Gd': 3, 'TA': 2, 'Fa': 1, 'Po': 0}
print('🟡 HeatingQC: --> Ordinal Encoding')
print('---'*20)
display(mapping_heatingqc)

In [ ]:
#CentralAir : Aire acondicionado central.
print('---'*20)
centralair_cat_all = ['Y', 'N']
centralair_cat_train = df_train['CentralAir'].unique().tolist()
missing_cats = list(set(centralair_cat_all) - set(centralair_cat_train))
print(f'{"🟢" if len(missing_cats) == 0 else "🔴"} missing categories {missing_cats}')

centralair_new_cat = df_train[~df_train['CentralAir'].isin(centralair_cat_all)]
centralair_new_cat = centralair_new_cat['CentralAir'].unique().tolist()
print(f'{"🟢" if len(centralair_new_cat) == 0 else "🔴"} new categories or diferent name -> {centralair_new_cat}')

#No se tiene un orden definido en las categorias
#Pocas categorias
#Visto que las categorias no tienen un orden definido, se procede a hacer one-hot-encoding
print('🟡 CentralAir: --> One-Hot-Encoding')
print('---'*20)
ohe_centralair = pd.get_dummies(df_train['CentralAir'], prefix='CentralAir', dtype=int)
display(ohe_centralair.head())

In [ ]:
#Electrical : Tipo de sistema eléctrico.
print('---'*20)
electrical_cat_all = ['SBrkr', 'FuseA', 'FuseF', 'FuseP', 'Mix']
electrical_cat_train = df_train['Electrical'].unique().tolist()
missing_cats = list(set(electrical_cat_all) - set(electrical_cat_train))
print(f'{"🟢" if len(missing_cats) == 0 else "🔴"} missing categories {missing_cats}')

electrical_new_cat = df_train[~df_train['Electrical'].isin(electrical_cat_all)]
electrical_new_cat = electrical_new_cat['Electrical'].unique().tolist()
print(f'{"🟢" if len(electrical_new_cat) == 0 else "🔴"} new categories or diferent name -> {electrical_new_cat}')

#No se tiene un orden definido en las categorias
#Pocas categorias
#Visto que las categorias no tienen un orden definido, se procede a hacer one-hot-encoding
#@@Puede ser aplicado con ordinal encoding

print('🟡 Electrical: --> One-Hot-Encoding')
print('---'*20)
ohe_electrical = pd.get_dummies(df_train['Electrical'], prefix='Electrical', dtype=int)
display(ohe_electrical.head())

In [ ]:
#KitchenQual : Calidad de la cocina.
print('---'*20)
kitchenqual_cat_all = ['Ex', 'Gd', 'TA', 'Fa', 'Po']
kitchenqual_cat_train = df_train['KitchenQual'].unique().tolist()
missing_cats = list(set(kitchenqual_cat_all) - set(kitchenqual_cat_train))
print(f'{"🟢" if len(missing_cats) == 0 else "🔴"} missing categories {missing_cats}')

kitchenqual_new_cat = df_train[~df_train['KitchenQual'].isin(kitchenqual_cat_all)]
kitchenqual_new_cat = kitchenqual_new_cat['KitchenQual'].unique().tolist()
print(f'{"🟢" if len(kitchenqual_new_cat) == 0 else "🔴"} new categories or diferent name -> {kitchenqual_new_cat}')

#Se define como ordinal encoding
#Pocas categorias
mapping_kitchenqual = {'Ex': 4, 'Gd': 3, 'TA': 2, 'Fa': 1, 'Po': 0}
print('🟡 KitchenQual: --> Ordinal Encoding')
print('---'*20)
display(mapping_kitchenqual)

In [ ]:
#Functional : Evaluación de la funcionalidad.
print('---'*20)
functional_cat_all = ['Typ', 'Min1', 'Min2', 'Mod', 'Maj1', 'Maj2', 'Sev', 'Sal']
functional_cat_train = df_train['Functional'].unique().tolist()
missing_cats = list(set(functional_cat_all) - set(functional_cat_train))
print(f'{"🟢" if len(missing_cats) == 0 else "🔴"} missing categories {missing_cats}')

functional_new_cat = df_train[~df_train['Functional'].isin(functional_cat_all)]
functional_new_cat = functional_new_cat['Functional'].unique().tolist()
print(f'{"🟢" if len(functional_new_cat) == 0 else "🔴"} new categories or diferent name -> {functional_new_cat}')

#Se define como ordinal encoding
#Pocas categorias
#Podria ser probado con one-hot-encoding
mapping_functional = {'Typ': 7, 'Min1': 6, 'Min2': 5, 'Mod': 4, 'Maj1': 3, 'Maj2': 2, 'Sev': 1, 'Sal': 0}
print('🟡 Functional: --> Ordinal Encoding')
print('---'*20)
display(mapping_functional)

In [ ]:
#GarageType : Ubicación del garaje.
print('---'*20)
garagetype_cat_all = [ "2Types", "Attchd", "Basment", "BuiltIn", "CarPort", "Detchd", "NA"]
garagetype_cat_train = df_train['GarageType'].unique().tolist()
missing_cats = list(set(garagetype_cat_all) - set(garagetype_cat_train))
print(f'{"🟢" if len(missing_cats) == 0 else "🔴"} missing categories {missing_cats}')

garagetype_new_cat = df_train[~df_train['GarageType'].isin(garagetype_cat_all)]
garagetype_new_cat = garagetype_new_cat['GarageType'].unique().tolist()
print(f'{"🟢" if len(garagetype_new_cat) == 0 else "🔴"} new categories or diferent name -> {garagetype_new_cat}')

#No se tiene un orden definido en las categorias
#Pocas categorias
#Visto que las categorias no tienen un orden definido, se procede a hacer one-hot-encoding

print('🟡 GarageType: --> One-Hot-Encoding')
print('---'*20)
ohe_garagetype = pd.get_dummies(df_train['GarageType'], prefix='GarageType', dtype=int)
display(ohe_garagetype.head())

In [ ]:
#GarageFinish : Acabado interior del garaje.
print('---'*20)
garagefinish_cat_all = ['Fin', 'RFn', 'Unf', 'NA']
garagefinish_cat_train = df_train['GarageFinish'].unique().tolist()
missing_cats = list(set(garagefinish_cat_all) - set(garagefinish_cat_train))
print(f'{"🟢" if len(missing_cats) == 0 else "🔴"} missing categories {missing_cats}')

garagefinish_new_cat = df_train[~df_train['GarageFinish'].isin(garagefinish_cat_all)]
garagefinish_new_cat = garagefinish_new_cat['GarageFinish'].unique().tolist()
print(f'{"🟢" if len(garagefinish_new_cat) == 0 else "🔴"} new categories or diferent name -> {garagefinish_new_cat}')

#Se define como ordinal encoding
#Pocas categorias
mapping_garagefinish = {'Fin': 3, 'RFn': 2, 'Unf': 1, 'NA': 0}
print('🟡 GarageFinish: --> Ordinal Encoding')
print('---'*20)
display(mapping_garagefinish)

In [ ]:
#GarageQual : Evaluación de la calidad del garaje.
print('---'*20)
garagequal_cat_all = ['Ex', 'Gd', 'TA', 'Fa', 'Po', 'NA']
garagequal_cat_train = df_train['GarageQual'].unique().tolist()
missing_cats = list(set(garagequal_cat_all) - set(garagequal_cat_train))
print(f'{"🟢" if len(missing_cats) == 0 else "🔴"} missing categories {missing_cats}')

garagequal_new_cat = df_train[~df_train['GarageQual'].isin(garagequal_cat_all)]
garagequal_new_cat = garagequal_new_cat['GarageQual'].unique().tolist()
print(f'{"🟢" if len(garagequal_new_cat) == 0 else "🔴"} new categories or diferent name -> {garagequal_new_cat}')

#Se define como ordinal encoding
#Pocas categorias
mapping_garagequal = {'Ex': 4, 'Gd': 3, 'TA': 2, 'Fa': 1, 'Po': 0, 'NA': 0}
print('🟡 GarageQual: --> Ordinal Encoding')
print('---'*20)
display(mapping_garagequal)

In [ ]:
#GarageCond : Evaluación de la condición del garaje.
print('---'*20)
garagecond_cat_all = ['Ex', 'Gd', 'TA', 'Fa', 'Po', 'NA']
garagecond_cat_train = df_train['GarageCond'].unique().tolist()
missing_cats = list(set(garagecond_cat_all) - set(garagecond_cat_train))
print(f'{"🟢" if len(missing_cats) == 0 else "🔴"} missing categories {missing_cats}')

garagecond_new_cat = df_train[~df_train['GarageCond'].isin(garagecond_cat_all)]
garagecond_new_cat = garagecond_new_cat['GarageCond'].unique().tolist()
print(f'{"🟢" if len(garagecond_new_cat) == 0 else "🔴"} new categories or diferent name -> {garagecond_new_cat}')

#Se define como ordinal encoding
#Pocas categorias
mapping_garagecond = {'Ex': 4, 'Gd': 3, 'TA': 2, 'Fa': 1, 'Po': 0, 'NA': 0}
print('🟡 GarageCond: --> Ordinal Encoding')
print('---'*20)
display(mapping_garagecond)

In [ ]:
#PavedDrive : Tipo de camino pavimentado.
print('---'*20)
paveddrive_cat_all = ['Y', 'P', 'N']
paveddrive_cat_train = df_train['PavedDrive'].unique().tolist()
missing_cats = list(set(paveddrive_cat_all) - set(paveddrive_cat_train))
print(f'{"🟢" if len(missing_cats) == 0 else "🔴"} missing categories {missing_cats}')

paveddrive_new_cat = df_train[~df_train['PavedDrive'].isin(paveddrive_cat_all)]
paveddrive_new_cat = paveddrive_new_cat['PavedDrive'].unique().tolist()
print(f'{"🟢" if len(paveddrive_new_cat) == 0 else "🔴"} new categories or diferent name -> {paveddrive_new_cat}')

#Se define como ordinal encoding
#Pocas categorias
mapping_paveddrive = {'Y': 1, 'P': 0, 'N': 0}
print('🟡 PavedDrive: --> Ordinal Encoding')
print('---'*20)
display(mapping_paveddrive)

In [ ]:
#SaleType : Tipo de venta.
print('---'*20)
saletype_cat_all = ['WD', 'CWD', 'VWD', 'New', 'COD', 'Con', 'ConLw', 'ConLI', 'ConLD', 'Oth']
saletype_cat_train = df_train['SaleType'].unique().tolist()
missing_cats = list(set(saletype_cat_all) - set(saletype_cat_train))
print(f'{"🟢" if len(missing_cats) == 0 else "🔴"} missing categories {missing_cats}')

saletype_new_cat = df_train[~df_train['SaleType'].isin(saletype_cat_all)]
saletype_new_cat = saletype_new_cat['SaleType'].unique().tolist()
print(f'{"🟢" if len(saletype_new_cat) == 0 else "🔴"} new categories or diferent name -> {saletype_new_cat}')

#No se tiene un orden definido en las categorias
#Pocas categorias
#Visto que las categorias no tienen un orden definido, se procede a hacer one-hot-encoding
print('🟡 SaleType: --> One-Hot-Encoding')
print('---'*20)
ohe_saletype = pd.get_dummies(df_train['SaleType'], prefix='SaleType', dtype=int)
display(ohe_saletype.head())

In [ ]:
#SaleCondition : Condición de la venta.
print('---'*20)
salecondition_cat_all = ['Normal', 'Abnorml', 'AdjLand', 'Alloca', 'Family', 'Partial']
salecondition_cat_train = df_train['SaleCondition'].unique().tolist()
missing_cats = list(set(salecondition_cat_all) - set(salecondition_cat_train))
print(f'{"🟢" if len(missing_cats) == 0 else "🔴"} missing categories {missing_cats}')

salecondition_new_cat = df_train[~df_train['SaleCondition'].isin(salecondition_cat_all)]
salecondition_new_cat = salecondition_new_cat['SaleCondition'].unique().tolist()
print(f'{"🟢" if len(salecondition_new_cat) == 0 else "🔴"} new categories or diferent name -> {salecondition_new_cat}')

#No se tiene un orden definido en las categorias
#Pocas categorias
#Visto que las categorias no tienen un orden definido, se procede a hacer one-hot-encoding
print('🟡 SaleCondition: --> One-Hot-Encoding')
print('---'*20)
ohe_salecondition = pd.get_dummies(df_train['SaleCondition'], prefix='SaleCondition', dtype=int)
display(ohe_salecondition.head())

In [ ]:
#MSSubClass: Clase de construcción.
print('---'*20)
msubclass_cat_all = [20, 30, 40, 45, 50, 60, 70, 75, 80, 85, 90, 120, 150, 160, 180, 190]
msubclass_cat_train = df_train['MSSubClass'].unique().tolist()
missing_cats = list(set(msubclass_cat_all) - set(msubclass_cat_train))
print(f'{"🟢" if len(missing_cats) == 0 else "🔴"} missing categories {missing_cats}')
msubclass_new_cat = df_train[~df_train['MSSubClass'].isin(msubclass_cat_all)]
msubclass_new_cat = msubclass_new_cat['MSSubClass'].unique().tolist()
print(f'{"🟢" if len(msubclass_new_cat) == 0 else "🔴"} new categories or diferent name -> {msubclass_new_cat}')
#No se tiene un orden definido en las categorias
#Visto que las categorias no tienen un orden definido, se procede a hacer one-hot-encoding
print('🟡 MSSubClass: --> One-Hot-Encoding')

In [ ]:
#LotArea: Tamaño del terreno.
print('---'*20)
#Es una variable numérica continua, no se aplica codificación

plt.figure(figsize=(30,4))
sns.boxplot(x=df_train['LotArea'], color='lightblue')
plt.title('Boxplot of LotArea');
df_train['LotArea'].describe().round(2)
quartil_one = np.percentile(df_train['LotArea'], 25)


def cal_quartiles(df_in):
    Q1 = np.percentile(df_in, 25)
    Q2 = np.percentile(df_in, 50)
    Q3 = np.percentile(df_in, 75)
    RIC = Q3 - Q1
    W1 = Q1 - 1.5 * RIC
    W1 = np.max(df_in[df_in >= W1]) if np.any(df_in >= W1) else np.min(df_in)
    W2 = Q1 + 1.5 * RIC
    W2 = np.min(df_in[df_in <= W2]) if np.any(df_in <= W2) else np.max(df_in)
    
    format_result = lambda x: f'{x:.2f}'
    
    return {
        "quartile_25": format_result(Q1),
        "quartile_50": format_result(Q2),
        "quartile_75": format_result(Q3),
        "RIC": format_result(RIC),
        "whisker_lower": format_result(W1),
        "whisker_upper": format_result(W2),
    }

data = cal_quartiles(df_train['LotArea'])
display(data)

##Analizar como metodo de la desviacion estandar
##prueba de commit desde pc de clai